# KPI Analysis – Seller Performance

This notebook analyzes seller performance based on sales, order volume, number of items sold, and customer review scores. The goal is to identify differences in seller performance and explore which sellers contribute most to overall marketplace activity.

In [1]:
import pandas as pd
import sys

print("Python:", sys.version)
print("Pandas:", pd.__version__)

Python: 3.13.15 (main, Sep  1 2026, 14:16:48) [MSC v.1944 64 bit (AMD64)]
Pandas: 3.0.5


In [2]:
from pathlib import Path

data_path = Path("../data")

csv_files = list(data_path.glob("*.csv"))

for file in csv_files:
    print(file.name)

In [3]:
from pathlib import Path
print(Path.cwd())

c:\Users\user777\Desktop\Conclusion Intelegence project\SpikupCapstone2026_CYRV\notebooks


In [4]:
data_path = Path("../data")

print("Data folder exists:", data_path.exists())

for item in data_path.iterdir():
    print(item.name)

Data folder exists: True
cyrv
README.md


In [5]:
data_path = Path("../data/cyrv")

csv_files = list(data_path.glob("*.csv"))

for file in csv_files:
    print(file.name)

CYRV_customers_dataset.csv
CYRV_geolocation_dataset.csv
CYRV_orders_dataset.csv
CYRV_order_items_dataset.csv
CYRV_order_payments_dataset.csv
CYRV_order_reviews_dataset.csv
CYRV_products_dataset.csv
CYRV_sellers_dataset.csv
product_category_name_translation.csv


In [6]:
file_info = []

for file in csv_files:
    file_info.append({
        "file": file.name,
        "size_mb": round(file.stat().st_size / (1024 * 1024), 2)
    })

pd.DataFrame(file_info).sort_values("size_mb", ascending=False)

,file,size_mb
1,CYRV_geolocation_dataset.csv,59.39
2,CYRV_orders_dataset.csv,16.93
3,CYRV_order_items_dataset.csv,14.83
5,CYRV_order_reviews_dataset.csv,13.78
0,CYRV_customers_dataset.csv,8.71
4,CYRV_order_payments_dataset.csv,5.61
6,CYRV_products_dataset.csv,2.30
7,CYRV_sellers_dataset.csv,0.17
8,product_category_name_translation.csv,0.00


In [7]:
customers = pd.read_csv(data_path / "CYRV_customers_dataset.csv")
orders = pd.read_csv(data_path / "CYRV_orders_dataset.csv")
order_items = pd.read_csv(data_path / "CYRV_order_items_dataset.csv")
payments = pd.read_csv(data_path / "CYRV_order_payments_dataset.csv")
reviews = pd.read_csv(data_path / "CYRV_order_reviews_dataset.csv")
products = pd.read_csv(data_path / "CYRV_products_dataset.csv")
sellers = pd.read_csv(data_path / "CYRV_sellers_dataset.csv")
category_translation = pd.read_csv(
    data_path / "product_category_name_translation.csv"
)

print("Datasets loaded successfully ✅")

Datasets loaded successfully ✅


In [8]:
orders.dtypes

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

In [9]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

orders[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [10]:
print("Order items columns:")
print(order_items.columns.tolist())

print("\nOrders columns:")
print(orders.columns.tolist())

print("\nReviews columns:")
print(reviews.columns.tolist())

print("\nSellers columns:")
print(sellers.columns.tolist())

Order items columns:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Orders columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Reviews columns:
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

Sellers columns:
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']


In [11]:
seller_performance = (
    order_items.groupby("seller_id")
    .agg(
        total_sales=("price", "sum"),
        number_of_orders=("order_id", "nunique"),
        number_of_items=("order_item_id", "count")
    )
    .reset_index()
)

seller_performance.head()

,seller_id,total_sales,number_of_orders,number_of_items
0,0015a82c2db000af6aaaf3ae2ecb0532,2685.00,3,3
1,001cca7ae9ae17fb1caed9dfb1094831,25080.03,200,239
2,001e6ad469a905060d959994f1b41e4f,250.00,1,1
3,002100f778ceb8431b7a1020ff7ab48f,1234.50,51,55
4,003554e2dce176b5555353e4f3555ac8,120.00,1,1


In [12]:
top_sellers_sales = (
    seller_performance
    .sort_values("total_sales", ascending=False)
    .head(10)
)

top_sellers_sales

,seller_id,total_sales,number_of_orders,number_of_items
857,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1132,1156
1013,53243585a1d6dc2643021fd1853d8905,222776.05,358,410
881,4a3ca9315b744ce9f8e9374361493884,200472.92,1806,1987
3024,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,585,586
1535,7c67e1448b00f6e969d365cea6b010ab,187923.89,982,1364
1560,7e93a43ef30c4f03f38b393420bc753a,176431.87,336,340
2643,da8622b14eb17ae2831f4ac5b9dab84a,160236.57,1314,1551
1505,7a67c85e85bb2ce8582c35f2203ad736,141745.53,1160,1171
192,1025f0e2d44d7041d6cf58b6550e0bfa,138968.55,915,1428
1824,955fee9216a65b617aa5c0531780ce60,135171.70,1287,1499


In [13]:
top_sellers_orders = (
    seller_performance
    .sort_values("number_of_orders", ascending=False)
    .head(10)
)

top_sellers_orders

,seller_id,total_sales,number_of_orders,number_of_items
1235,6560211a19b47992c3666cc44a7e94c0,123304.83,1854,2033
881,4a3ca9315b744ce9f8e9374361493884,200472.92,1806,1987
2481,cc419e0650a3c5ba77189a1882b7556a,104288.42,1706,1775
368,1f50f920176fa81dab994f9023523100,106939.21,1404,1931
2643,da8622b14eb17ae2831f4ac5b9dab84a,160236.57,1314,1551
1824,955fee9216a65b617aa5c0531780ce60,135171.70,1287,1499
1505,7a67c85e85bb2ce8582c35f2203ad736,141745.53,1160,1171
2836,ea8482cd71df3c1969d7b9473ff13abc,37177.52,1146,1203
857,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1132,1156
731,3d871de0142ce09b7081e2b9d1733cb1,94914.20,1080,1147


In [14]:
seller_reviews = (
    order_items[["order_id", "seller_id"]]
    .drop_duplicates()
    .merge(
        reviews[["order_id", "review_score"]],
        on="order_id",
        how="inner"
    )
)

seller_review_scores = (
    seller_reviews.groupby("seller_id")
    .agg(
        average_review_score=("review_score", "mean"),
        number_of_reviews=("review_score", "count")
    )
    .reset_index()
)

seller_review_scores.head()

,seller_id,average_review_score,number_of_reviews
0,0015a82c2db000af6aaaf3ae2ecb0532,3.666667,3
1,001cca7ae9ae17fb1caed9dfb1094831,3.984772,197
2,001e6ad469a905060d959994f1b41e4f,1.000000,1
3,002100f778ceb8431b7a1020ff7ab48f,3.903846,52
4,003554e2dce176b5555353e4f3555ac8,5.000000,1


In [15]:
seller_analysis = seller_performance.merge(
    seller_review_scores,
    on="seller_id",
    how="left"
)

seller_analysis.head()

,seller_id,total_sales,number_of_orders,number_of_items,average_review_score,number_of_reviews
0,0015a82c2db000af6aaaf3ae2ecb0532,2685.00,3,3,3.666667,3.0
1,001cca7ae9ae17fb1caed9dfb1094831,25080.03,200,239,3.984772,197.0
2,001e6ad469a905060d959994f1b41e4f,250.00,1,1,1.000000,1.0
3,002100f778ceb8431b7a1020ff7ab48f,1234.50,51,55,3.903846,52.0
4,003554e2dce176b5555353e4f3555ac8,120.00,1,1,5.000000,1.0


In [16]:
seller_analysis.describe()

,total_sales,number_of_orders,number_of_items,average_review_score,number_of_reviews
count,3095.000000,3095.000000,3095.000000,3090.000000,3090.000000
mean,4391.484233,32.313409,36.397415,3.998713,32.300647
std,13921.997192,105.139763,119.193461,0.955220,105.196143
min,3.500000,1.000000,1.000000,1.000000,1.000000
25%,208.850000,2.000000,2.000000,3.773990,2.000000
50%,821.480000,6.000000,8.000000,4.186970,6.000000
75%,3280.830000,21.500000,24.000000,4.600000,21.000000
max,229472.630000,1854.000000,2033.000000,5.000000,1844.000000


In [17]:
reliable_sellers = seller_analysis[
    seller_analysis["number_of_reviews"] >= 20
].copy()

print("Sellers with at least 20 reviews:", len(reliable_sellers))

Sellers with at least 20 reviews: 811


In [18]:
top_rated_sellers = (
    reliable_sellers
    .sort_values(
        ["average_review_score", "number_of_reviews"],
        ascending=[False, False]
    )
    .head(10)
)

top_rated_sellers

,seller_id,total_sales,number_of_orders,number_of_items,average_review_score,number_of_reviews
863,48efc9d94a9834137efd9ea76b065a38,345.60,33,34,5.000000,33.0
786,41c2bad7229b0c25e6becf179ebf63ff,782.00,20,23,4.950000,20.0
35,02f5837340d7eb4f653d676c7256523a,3883.05,30,30,4.833333,30.0
2531,d13e50eaa47b4cbe9eb81465865d8cfc,7155.05,67,69,4.820896,67.0
799,42fa4ee7240e9b8eb4576358ec142ba7,3845.70,22,22,4.809524,21.0
1624,83e197e95a1bbabc8c75e883ed016c47,6693.69,47,55,4.808511,47.0
2629,d9bd94811c3338dceb4181f3dbc0c73e,6911.02,54,61,4.796296,54.0
1515,7ade73f1b9b4e965f9009a4c3a7e2c15,1880.30,27,28,4.769231,26.0
1131,5bffbafbb22daf6d3bfc216a46db8708,5239.70,21,23,4.761905,21.0
651,3785b653b1b82de85ab47dd139938091,1552.00,21,24,4.727273,22.0


In [19]:
top_sales_reliable = (
    reliable_sellers
    .sort_values("total_sales", ascending=False)
    .head(10)
)

top_sales_reliable

,seller_id,total_sales,number_of_orders,number_of_items,average_review_score,number_of_reviews
857,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1132,1156,4.133452,1124.0
1013,53243585a1d6dc2643021fd1853d8905,222776.05,358,410,4.132022,356.0
881,4a3ca9315b744ce9f8e9374361493884,200472.92,1806,1987,3.827873,1801.0
3024,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,585,586,4.339071,581.0
1535,7c67e1448b00f6e969d365cea6b010ab,187923.89,982,1364,3.488798,982.0
1560,7e93a43ef30c4f03f38b393420bc753a,176431.87,336,340,4.214925,335.0
2643,da8622b14eb17ae2831f4ac5b9dab84a,160236.57,1314,1551,4.178868,1325.0
1505,7a67c85e85bb2ce8582c35f2203ad736,141745.53,1160,1171,4.244156,1155.0
192,1025f0e2d44d7041d6cf58b6550e0bfa,138968.55,915,1428,3.989059,914.0
1824,955fee9216a65b617aa5c0531780ce60,135171.70,1287,1499,4.158718,1279.0


In [20]:
seller_correlation = reliable_sellers[
    ["total_sales", "number_of_orders", "average_review_score"]
].corr()

seller_correlation.round(2)

,total_sales,number_of_orders,average_review_score
total_sales,1.00,0.76,-0.03
number_of_orders,0.76,1.00,-0.04
average_review_score,-0.03,-0.04,1.00


In [21]:
sales_median = reliable_sellers["total_sales"].median()
review_median = reliable_sellers["average_review_score"].median()

print("Median sales:", round(sales_median, 2))
print("Median review score:", round(review_median, 2))

Median sales: 6455.4
Median review score: 4.15


In [22]:
strong_sellers = reliable_sellers[
    (reliable_sellers["total_sales"] >= sales_median) &
    (reliable_sellers["average_review_score"] >= review_median)
].copy()

print("Strong sellers:", len(strong_sellers))

strong_sellers.sort_values(
    "total_sales", ascending=False
).head(10)

Strong sellers: 199


,seller_id,total_sales,number_of_orders,number_of_items,average_review_score,number_of_reviews
3024,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,585,586,4.339071,581.0
1560,7e93a43ef30c4f03f38b393420bc753a,176431.87,336,340,4.214925,335.0
2643,da8622b14eb17ae2831f4ac5b9dab84a,160236.57,1314,1551,4.178868,1325.0
1505,7a67c85e85bb2ce8582c35f2203ad736,141745.53,1160,1171,4.244156,1155.0
1824,955fee9216a65b617aa5c0531780ce60,135171.70,1287,1499,4.158718,1279.0
843,46dc3b2cc0980fb8ec44634e21d2718e,128111.19,521,542,4.194175,515.0
1198,620c87c171fb2a6dd6e8bb4dec959fc6,114774.50,740,798,4.251691,739.0
1954,a1043bafd471dff536d0c462352beb48,101901.16,718,770,4.222378,715.0
2871,edb1ef5e36e0c8cd84eb3c9b003e486d,79284.55,166,175,4.421687,166.0
2489,ccc4bbb5f32a6ab2b7066a4130f114e3,74004.62,187,192,4.284946,186.0


## Seller Performance – Key Findings

- Seller performance varies substantially across the marketplace, with a relatively small group generating very high sales and order volumes.
- Sales and order volume are strongly related (correlation = 0.76), indicating that higher sales are mainly associated with processing more orders.
- Customer review scores show almost no correlation with sales (-0.03) or order volume (-0.04). High sales therefore do not necessarily imply higher customer satisfaction.
- Among sellers with at least 20 reviews, the median sales value is 6,455.40 and the median review score is 4.15.
- 199 sellers perform above or equal to both median thresholds, combining relatively strong sales performance with strong customer satisfaction.